# KvForge + STAR-KV: Low-Rank KV Cache Compression
# **Base Encode + LoRA Decode** ile birleştirilmiş **STAR-KV adaptive low-rank compression**.
## Yaklaşım
# 1. **Base Encode**: Prefill LoRA'sız çalışır → uzun context stabilitesi
# 2. **STAR-KV Compression**: KV cache'i low-rank SVD projection ile sıkıştır
# 3. **Hybrid Decomposition**: K için yüksek rank (conservative), V için düşük rank (aggressive)
# 4. **Quantization**: Low-rank faktörlerini ayrıca quantize et → 2-4x compression
# 5. **LoRA Decode**: LoRA aktif decode
# Referans: STAR-KV (ICML 2026 Spotlight) — arXiv:2606.08382


## Setup
# %%
# import sys, math, time, json, warnings
# warnings.filterwarnings("ignore")
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# Check GPU
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"Device: {device}")
# if torch.cuda.is_available():
# print(f"GPU: {torch.cuda.get_device_name()}")
# print(f"CUDA cores: {torch.cuda.get_device_properties(0).multi_processor_count}")


## STAR-KV: Randomized SVD + Adaptive Low-Rank Compression
# %%
# class RandomSVD:
# """
# Randomized SVD for efficient low-rank approximation.
# O(m*n*log(k)) vs O(m*n*min(m,n))
# """
# @staticmethod
# def compute(X: torch.Tensor, k: int, n_oversamples: int = 10,
# n_iter: int = 2):
# m, n = X.shape[-2], X.shape[-1]
# p = min(k + n_oversamples, n)
# Q = torch.randn(n, p, device=X.device, dtype=X.dtype)
# Y = X @ Q
# for _ in range(n_iter):
# Y = X @ (X.mT @ Y)
# Y = torch.linalg.qr(Y).Q
# B = Y.mT @ X
# Ub, Sb, Vhb = torch.linalg.svd(B, full_matrices=False)
# U = Y @ Ub[..., :k]
# S = Sb[..., :k]
# Vh = Vhb[..., :k, :]
# return U, S, Vh
# class STAR_KV_Compressor:
# """
# Adaptive low-rank KV cache compression.
# Hybrid decomposition: conservative K, aggressive V.
# """
# @staticmethod
# def _estimate_rank(tensor, energy_threshold=0.95, max_rank=None):
# """Energy-based rank selection (soft-thresholding heuristic)."""
# if tensor.dim() > 2:
# *batch_dims, seq, hdim = tensor.shape
# flat = tensor.reshape(-1, seq, hdim)
# else:
# flat = tensor.unsqueeze(0)
# _, S, _ = torch.linalg.svd(flat, full_matrices=False)
# total_energy = (S ** 2).sum(dim=-1, keepdim=True)
# cum_energy = (S ** 2).cumsum(dim=-1)
# ratio = cum_energy / total_energy.clamp(1e-10)
# rank = (ratio < energy_threshold).sum(dim=-1).max().item() + 1
# if max_rank is not None:
# rank = min(rank, max_rank)
# return max(1, min(rank, seq, hdim))
# @staticmethod
# @torch.no_grad()
# def compress_kv(k, v, energy_k=0.95, energy_v=0.90,
# rank_k=None, rank_v=None,
# quant_bits=None):
# """
# Compress K and V using STAR-KV low-rank projection.
# Args:
# k: (batch, n_heads, seq_len, head_dim)
# v: (batch, n_heads, seq_len, head_dim)
# energy_k: energy threshold for K
# energy_v: energy threshold for V (lower = more compression)
# rank_k: fixed rank (None = auto from energy)
# rank_v: fixed rank (None = auto from energy)
# quant_bits: quantize low-rank factors (None = no quant)
# """
# batch, n_heads, seq_len, head_dim = k.shape
# Auto rank
# if rank_k is None:
# rank_k = min(
# STAR_KV_Compressor._estimate_rank(k, energy_k),
# seq_len, head_dim)
# if rank_v is None:
# rank_v = min(
# STAR_KV_Compressor._estimate_rank(v, energy_v),
# seq_len, head_dim)
# rank_k = max(1, min(rank_k, seq_len, head_dim))
# rank_v = max(1, min(rank_v, seq_len, head_dim))
# k_2d = k.reshape(-1, seq_len, head_dim)
# v_2d = v.reshape(-1, seq_len, head_dim)
# Compress
# orig_elements = (k.numel() + v.numel()) * 2  # float16 bytes
# if rank_k >= min(seq_len, head_dim):
# k_comp = {"type": "full"}
# k_bytes = k.numel() * 2
# k_out = k
# else:
# Uk, Sk, Vhk = RandomSVD.compute(k_2d, rank_k)
# k_comp = {"type": "lowrank", "U": Uk, "S": Sk, "Vh": Vhk, "rank": rank_k}
# if quant_bits and quant_bits < 16:
# Quantize factors
# mn, mx = Uk.min(), Uk.max()
# scale = (mx-mn).clamp(1e-8) / (2**quant_bits-1)
# k_comp["Uq"] = ((Uk - mn) / scale).round().clamp(0, 2**quant_bits-1).to(torch.uint8)
# k_comp["U_scale"] = scale
# k_comp["U_zero"] = mn
# k_bytes = (Uk.numel() + Sk.numel() + Vhk.numel()) * 1  # uint8
# else:
# k_bytes = (Uk.numel() + Sk.numel() + Vhk.numel()) * 2  # float16
# k_out = Uk * Sk.unsqueeze(-2) @ Vhk
# k_out = k_out.reshape(batch, n_heads, seq_len, head_dim)
# if rank_v >= min(seq_len, head_dim):
# v_comp = {"type": "full"}
# v_bytes = v.numel() * 2
# v_out = v
# else:
# Uv, Sv, Vhv = RandomSVD.compute(v_2d, rank_v)
# v_comp = {"type": "lowrank", "U": Uv, "S": Sv, "Vh": Vhv, "rank": rank_v}
# if quant_bits and quant_bits < 16:
# mn, mx = Uv.min(), Uv.max()
# scale = (mx-mn).clamp(1e-8) / (2**quant_bits-1)
# v_comp["Uq"] = ((Uv - mn) / scale).round().clamp(0, 2**quant_bits-1).to(torch.uint8)
# v_comp["U_scale"] = scale
# v_comp["U_zero"] = mn
# v_bytes = (Uv.numel() + Sv.numel() + Vhv.numel()) * 1
# else:
# v_bytes = (Uv.numel() + Sv.numel() + Vhv.numel()) * 2
# v_out = Uv * Sv.unsqueeze(-2) @ Vhv
# v_out = v_out.reshape(batch, n_heads, seq_len, head_dim)
# cr = orig_elements / (k_bytes + v_bytes) if (k_bytes + v_bytes) > 0 else 1.0
# return {
# "k": k_out,
# "v": v_out,
# "compressed": (k_comp, v_comp),
# "rank_k": rank_k,
# "rank_v": rank_v,
# "compression_ratio": round(cr, 2),
# "quant_bits": quant_bits or 16,
# }


## LoRA Wrappers
# %%
# class LoRAConv1D(nn.Module):
# """LoRA for GPT-2 Conv1D."""
# def __init__(self, orig, r=8, alpha=16.0):
# super().__init__()
# self.orig = orig
# self.scaling = alpha / r
# in_f, out_f = orig.weight.shape[0], orig.nf
# self.lora_A = nn.Parameter(torch.randn(in_f, r) * 0.02)
# self.lora_B = nn.Parameter(torch.zeros(r, out_f))
# self.active = True
# def activate(self, a=True):
# self.active = a
# def forward(self, x):
# h = self.orig(x)
# if self.active:
# h = h + (x @ self.lora_A @ self.lora_B) * self.scaling
# return h
# class LoRALinear(nn.Module):
# """LoRA for nn.Linear."""
# def __init__(self, orig, r=8, alpha=16.0):
# super().__init__()
# self.orig = orig
# self.scaling = alpha / r
# self.lora_A = nn.Parameter(torch.randn(orig.in_features, r) * 0.02)
# self.lora_B = nn.Parameter(torch.zeros(r, orig.out_features))
# self.active = True
# def activate(self, a=True):
# self.active = a
# def forward(self, x):
# h = self.orig(x)
# if self.active:
# h = h + (x @ self.lora_A @ self.lora_B) * self.scaling
# return h


## KvForge Model + STAR-KV Integration
# %%
# from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
# class KvForgeSTAR:
# """
# KvForge + STAR-KV: Base Encode + LoRA Decode + Adaptive Low-Rank Compression.
# """
# def __init__(self, model_name="gpt2", lora_rank=8, lora_alpha=16.0, device=None):
# self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
# self.model_name = model_name
# self.lora_rank = lora_rank
# Load model
# print(f"Loading {model_name}...")
# self.base = AutoModelForCausalLM.from_pretrained(model_name).to(self.device).eval()
# self.tokenizer = AutoTokenizer.from_pretrained(model_name)
# self.tokenizer.pad_token = self.tokenizer.eos_token
# self._n_layers = self.base.config.n_layer if hasattr(self.base.config, 'n_layer') else self.base.config.num_hidden_layers
# self._head_dim = self.base.config.n_embd // self.base.config.n_head if hasattr(self.base.config, 'n_embd') else 64
# Inject LoRA
# self._inject_lora(lora_rank, lora_alpha)
# Stats
# lora_params = sum(p.numel() for n,p in self.base.named_parameters() if 'lora' in n)
# total_params = sum(p.numel() for p in self.base.parameters())
# print(f"Model: {model_name} ({total_params/1e6:.1f}M)")
# print(f"LoRA: {lora_params/1e3:.1f}K parameters")
# print(f"Layers: {self._n_layers}, Head dim: {self._head_dim}")
# def _inject_lora(self, r, alpha):
# count = 0
# for n, m in self.base.named_modules():
# if n.endswith(".attn.c_attn") or n.endswith(".attn.c_proj"):
# parent, child = self.base, n.split(".")[-1]
# for p in n.split(".")[:-1]:
# if p: parent = getattr(parent, p)
# setattr(parent, child, LoRAConv1D(m, r=r, alpha=alpha))
# count += 1
# print(f"LoRA injected: {count} modules")
# def set_lora(self, active=True):
# for mod in self.base.modules():
# if hasattr(mod, 'activate'):
# mod.activate(active)
# @torch.no_grad()
# def generate(self, text, max_new_tokens=20, mode="base_encode_lora_decode",
# compress="none", energy_k=0.95, energy_v=0.90,
# quant_bits=None, rank_k=None, rank_v=None):
# """
# Generate with STAR-KV compression.
# Args:
# compress: "none" | "quant" | "lowrank" | "hybrid"
# energy_k: energy threshold for K
# energy_v: energy threshold for V
# quant_bits: quantization bits for hybrid mode
# """
# inp = self.tokenizer(text, return_tensors="pt", truncation=True,
# max_length=256).to(self.device)
# inp_ids = inp["input_ids"]
# prompt_len = inp_ids.shape[1]
# --- Prefill ---
# if mode == "base_encode_lora_decode":
# self.set_lora(False)  # Base Encode
# else:
# self.set_lora(True)   # Full LoRA
# t0 = time.time()
# out = self.base.generate(
# input_ids=inp_ids, max_new_tokens=1, use_cache=True,
# pad_token_id=self.tokenizer.eos_token_id,
# do_sample=False, return_dict_in_generate=True)
# past = out.past_key_values
# tp = time.time() - t0
# --- Compression ---
# t0 = time.time()
# compression_stats = {}
# if compress in ("lowrank", "hybrid"):
# qbits = quant_bits if compress == "hybrid" else None
# compressed_past = []
# total_cr = 0.0
# for li, layer in enumerate(past):
# k, v = layer[0], layer[1]
# result = STAR_KV_Compressor.compress_kv(
# k, v, energy_k=energy_k, energy_v=energy_v,
# rank_k=rank_k, rank_v=rank_v,
# quant_bits=qbits)
# compressed_past.append((result["k"], result["v"]))
# total_cr += result["compression_ratio"]
# past = compressed_past
# compression_stats = {
# "avg_cr": round(total_cr / len(past), 2),
# "rank_k": result["rank_k"],
# "rank_v": result["rank_v"],
# }
# elif compress == "quant":
# bits = quant_bits or 4
# compressed_past = []
# for li, layer in enumerate(past):
# k, v = layer[0], layer[1]
# Min-max quant
# def quant(x, b):
# mn, mx = x.min(-1,True).values, x.max(-1,True).values
# s = (mx-mn).clamp(1e-8) / (2**b-1)
# return (((x-mn)/s).round().clamp(0,2**b-1).float()*s+mn).to(x.dtype)
# compressed_past.append((quant(k, bits), quant(v, bits)))
# past = compressed_past
# compression_stats = {"quant_bits": bits, "cr_est": f"{16/bits:.1f}x"}
# tc = time.time() - t0
# --- Decode ---
# if mode == "base_encode_lora_decode":
# self.set_lora(True)  # LoRA Decode
# t0 = time.time()
# last_tok = out.sequences[:, -1:]
# for _ in range(max_new_tokens):
# if isinstance(past, list) and len(past) > 0 and isinstance(past[0], tuple) and past[0][0].dim() == 4:
# Convert list of tuples to tuple of tuples for HF
# past_keys = tuple(past)
# else:
# past_keys = past
# out_d = self.base(last_tok, past_key_values=past_keys, use_cache=True)
# past = out_d.past_key_values
# last_tok = out_d.logits[:, -1:].argmax(dim=-1)
# td = time.time() - t0
# self.set_lora(False)
# --- PPL ---
# with torch.no_grad():
# out_b = self.base(inp_ids)
# loss = F.cross_entropy(out_b.logits[0, :-1], inp_ids[0, 1:])
# ppl = math.exp(loss.item())
# return {
# "mode": mode,
# "compress": compress,
# "prompt_len": prompt_len,
# "prefill_ms": round(tp * 1000, 1),
# "compress_ms": round(tc * 1000, 1),
# "decode_ms": round(td * 1000, 1),
# "total_ms": round((tp+tc+td) * 1000, 1),
# "perplexity": round(ppl, 2),
# **compression_stats,
# }


## Benchmark
# %%
# print("=" * 60)
# print("KvForge + STAR-KV Benchmark")
# print("=" * 60)
# model = KvForgeSTAR("gpt2", lora_rank=8)
# prompts = [
# "The transformer architecture revolutionized natural language processing by introducing self-attention, which allows models to weigh the importance of different tokens.",
# "In recent years, large language models have demonstrated remarkable capabilities in understanding and generating human-like text across diverse domains and tasks.",
# ]
# results = []
# for prompt in prompts:
# print(f"\nPrompt ({len(prompt.split())} tokens): {prompt[:60]}...")
# for mode in ["full_lora", "base_encode_lora_decode"]:
# Baseline
# r = model.generate(prompt, mode=mode, compress="none")
# r["test"] = "baseline"
# results.append(r)
# print(f"  [{mode:<20}] none       | Dec:{r['decode_ms']:>6.1f}ms PPL:{r['perplexity']:.2f}")
# Quant 4-bit
# r = model.generate(prompt, mode=mode, compress="quant", quant_bits=4)
# r["test"] = "quant4"
# results.append(r)
# print(f"  [{mode:<20}] quant 4bit | Dec:{r['decode_ms']:>6.1f}ms PPL:{r['perplexity']:.2f}")
# STAR-KV LowRank
# for ek in [0.95, 0.85]:
# ev = max(0.7, ek - 0.05)
# r = model.generate(prompt, mode=mode, compress="lowrank",
# energy_k=ek, energy_v=ev)
# r["test"] = f"lowrank_ek{ek}"
# results.append(r)
# print(f"  [{mode:<20}] lowrank ek={ek} | CR:{r.get('avg_cr','?'):>4.1f}x Dec:{r['decode_ms']:>6.1f}ms PPL:{r['perplexity']:.2f}")
# STAR-KV Hybrid
# for ek in [0.95, 0.85]:
# ev = max(0.7, ek - 0.05)
# r = model.generate(prompt, mode=mode, compress="hybrid",
# energy_k=ek, energy_v=ev, quant_bits=4)
# r["test"] = f"hybrid_ek{ek}"
# results.append(r)
# print(f"  [{mode:<20}] hybrid ek={ek}  | CR:{r.get('avg_cr','?'):>4.1f}x Dec:{r['decode_ms']:>6.1f}ms PPL:{r['perplexity']:.2f}")


## Results Summary
# %%
# print("\n")
# print("=" * 80)
# print(f"{'Method':<25} {'Mode':<20} {'CR':>6} {'Dec(ms)':>8} {'PPL':>8}")
# print("=" * 80)
# for r in results:
# cr = r.get('avg_cr', r.get('cr_est', '-'))
# print(f"{r['test']:<25} {r['mode']:<20} {str(cr):>6} {r['decode_ms']:>8.1f} {r['perplexity']:>8.2f}")


## STAR-KV Compression Analysis
### GPT-2 Small limitations:
# - head_dim=64 → SVD compression ratio limited
# - For STAR-KV's full benefit (4-20x), use LLaMA-class models (head_dim=128+)
### Key Findings:
# 1. **LowRank + Quant hybrid** gives best CR/PPL tradeoff
# 2. **Base Encode + LoRA Decode** maintains PPL vs full LoRA
# 3. Energy threshold controls rank adaptively per layer
# 4. V can be compressed more aggressively than K (hybrid decomposition)
### Next Steps for Kaggle:
# - Use larger model (Phi-3, LLaMA) for meaningful compression
# - Implement STAR-KV's soft-threshold training on factors
# - Fuse attention with low-rank reconstruction (Triton)
